In [ ]:
# --- Xray Cloud: Release Test Report (Dashboard-Accurate: Unique + Raw Execution, Compatible Schema) ---

import requests, getpass, csv, os, datetime
from collections import Counter, defaultdict

# ----------------------------
# Config
# ----------------------------
PLAN_KEY = "PROJ-1001"   # <-- set your Test Plan key here

XRAY_AUTH_URL    = "https://xray.cloud.getxray.app/api/v2/authenticate"
XRAY_GRAPHQL_URL = "https://xray.cloud.getxray.app/api/v2/graphql"

VERIFY_SSL = False
if not VERIFY_SSL:
    from urllib3.exceptions import InsecureRequestWarning
    import urllib3; urllib3.disable_warnings(InsecureRequestWarning)

# ----------------------------
# Auth
# ----------------------------
CID  = getpass.getpass("XRAY_CLIENT_ID: ")
CSEC = getpass.getpass("XRAY_CLIENT_SECRET: ")

def auth():
    r = requests.post(XRAY_AUTH_URL, json={"client_id": CID, "client_secret": CSEC},
                      timeout=30, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"Auth failed: {r.status_code} {r.text[:300]}")
    return r.text.strip('"')

TOKEN   = auth()
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

def gql(query, variables=None):
    r = requests.post(XRAY_GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                      headers=HEADERS, timeout=90, verify=VERIFY_SSL)
    if r.status_code != 200:
        raise SystemExit(f"GraphQL HTTP {r.status_code}: {r.text[:300]}")
    data = r.json()
    if "errors" in data:
        raise SystemExit(f"GraphQL errors: {data['errors']}")
    return data["data"]

# ----------------------------
# Status normalization
# ----------------------------
STATUS_MAP = {
    "PASSED": "PASSED", "PASS": "PASSED", "OK": "PASSED",
    "FAILED": "FAILED", "FAIL": "FAILED",
    "TO DO": "TO DO", "TODO": "TO DO", "NOT_RUN": "TO DO", "UNTESTED": "TO DO",
    "BLOCKED": "BLOCKED",
    "EXECUTING": "EXECUTING", "IN PROGRESS": "EXECUTING",
    "DESCOPED": "DESCOPED", "NOT EXECUTED (DESCOPED)": "DESCOPED",
    "SKIPPED (DESCOPED)": "DESCOPED", "SKIPPED": "DESCOPED",
    "WONT DO": "DESCOPED", "DECLINED": "DESCOPED",
}
def norm_status(s: str) -> str:
    return STATUS_MAP.get((s or "").strip().upper(), s)

# ----------------------------
# 1) Resolve Test Plan metadata
# ----------------------------
Q_PLANS = """
query($jql:String!, $limit:Int!, $start:Int!) {
  getTestPlans(jql:$jql, limit:$limit, start:$start) {
    results { issueId jira(fields:["key","summary"]) }
  }
}
"""
plans = gql(Q_PLANS, {"jql": f'key = "{PLAN_KEY}"', "limit": 1, "start": 0})
if not plans["getTestPlans"]["results"]:
    raise SystemExit(f"No Test Plan found for key {PLAN_KEY}")
PLAN_ID = plans["getTestPlans"]["results"][0]["issueId"]
PLAN_NAME = f'{plans["getTestPlans"]["results"][0]["jira"]["key"]} - {plans["getTestPlans"]["results"][0]["jira"]["summary"]}'

# ----------------------------
# 2) Get PLANNED tests (ground truth for plan scope)
# ----------------------------
Q_PLAN_TESTS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    tests(limit:$limit, start:$start) {
      total start limit
      results { jira(fields:["key"]) }
    }
  }
}
"""
planned_keys = set()
start = 0
page = 300
while True:
    d = gql(Q_PLAN_TESTS, {"planId": PLAN_ID, "limit": page, "start": start})
    block = d["getTestPlan"]["tests"]
    for t in block["results"]:
        key = (t.get("jira") or {}).get("key")
        if key:
            planned_keys.add(key)
    start += block["limit"]
    if start >= block["total"]:
        break
planned_count = len(planned_keys)

# ----------------------------
# 3) Collect all Test Executions under the plan
# ----------------------------
Q_PLAN_EXECS = """
query($planId:String!, $limit:Int!, $start:Int!) {
  getTestPlan(issueId:$planId) {
    testExecutions(limit:$limit, start:$start) {
      total start limit
      results { issueId }
    }
  }
}
"""
exec_ids = []
start = 0
page = 100
while True:
    d = gql(Q_PLAN_EXECS, {"planId": PLAN_ID, "limit": page, "start": start})
    te = d["getTestPlan"]["testExecutions"]
    exec_ids += [x["issueId"] for x in te["results"]]
    start += te["limit"]
    if start >= te["total"]:
        break

# ----------------------------
# 4) Pull all Test Runs for those executions
# ----------------------------
Q_RUNS = """
query($execIds:[String], $limit:Int!, $start:Int!) {
  getTestRuns(testExecIssueIds:$execIds, limit:$limit, start:$start) {
    total start limit
    results {
      status { name }
      startedOn
      test { jira(fields:["key"]) }
    }
  }
}
"""
runs_by_test = defaultdict(list)
total_runs = 0

if exec_ids:
    BATCH = 50
    for i in range(0, len(exec_ids), BATCH):
        batch = exec_ids[i:i+BATCH]
        start = 0
        limit = 500
        while True:
            data = gql(Q_RUNS, {"execIds": batch, "limit": limit, "start": start})
            gr = data["getTestRuns"]
            for tr in gr["results"]:
                key = (tr.get("test") or {}).get("jira", {}).get("key")
                status = norm_status((tr.get("status") or {}).get("name"))
                ts = tr.get("startedOn") or ""
                if key:
                    runs_by_test[key].append({"status": status, "ts": ts})
                    total_runs += 1
            start += gr["limit"]
            if start >= gr["total"]:
                break

# ----------------------------
# 5) Derive latest (unique) status per PLANNED test
#    - tests with no runs -> "TO DO"
# ----------------------------
latest_status = {}
for key in planned_keys:
    if key in runs_by_test and runs_by_test[key]:
        latest = sorted(runs_by_test[key], key=lambda x: x["ts"] or "", reverse=True)[0]
        latest_status[key] = latest["status"]
    else:
        latest_status[key] = "TO DO"

unique_counts = Counter(latest_status.values())

# ----------------------------
# 6) Duplicate runs = extra run instances beyond the first for tests that ran
#    (does NOT penalize tests with zero runs)
# ----------------------------
duplicate_runs = 0
for key, items in runs_by_test.items():
    if len(items) > 1:
        duplicate_runs += (len(items) - 1)

# ----------------------------
# 7) Derived metrics (dashboard-style)
# ----------------------------
passed = unique_counts.get("PASSED", 0)
failed = unique_counts.get("FAILED", 0)
executing = unique_counts.get("EXECUTING", 0)
blocked = unique_counts.get("BLOCKED", 0)
descoped = unique_counts.get("DESCOPED", 0)
todo = unique_counts.get("TO DO", 0)

executed = planned_count - todo - descoped
pass_pct = round((passed / executed * 100.0), 2) if executed else 0.0
completion_pct = round((executed / planned_count * 100.0), 2) if planned_count else 0.0
remaining_pct = round(((todo + descoped) / planned_count * 100.0), 2) if planned_count else 0.0

# ----------------------------
# 8) Print Markdown summary
# ----------------------------
print(f"\n### Test Plan Rollup — {PLAN_NAME}\n")
print(f"**Planned (unique tests): {planned_count}**\n")

print("#### Unique Test Status (latest run per test; matches Xray dashboard)")
print("| Status | Count |")
print("|---|---|")
for s in ["PASSED", "FAILED", "EXECUTING", "BLOCKED", "DESCOPED", "TO DO"]:
    print(f"| {s} | {unique_counts.get(s,0)} |")

print("")
print("| Metric | Value |")
print("|---|---|")
print(f"| Pass % (of executed) | {pass_pct}% |")
print(f"| RT Completion % | {completion_pct}% |")
print(f"| RT Remaining % | {remaining_pct}% |")

print("\n#### Execution Effort (All Runs)")
print("| Metric | Count |")
print("|---|---|")
print(f"| Actual Execution (Raw Runs) | {total_runs} |")
print(f"| Duplicate Runs | {duplicate_runs} |")
print("")

# ----------------------------
# 9) Save daily CSV snapshot
# ----------------------------
def save_daily_snapshot(plan_key, planned, unique_c, total_runs, dup_runs,
                        pass_pct, completion_pct, remaining_pct):
    today = datetime.datetime.now().strftime("%Y-%m-%d")
    fname = f"Xray_Release Test Report_{plan_key}.csv"
    row = {
        "Date": today,
        "Planned": planned,
        "Unique Passed": unique_c.get("PASSED", 0),
        "Unique Failed": unique_c.get("FAILED", 0),
        "Unique Executing": unique_c.get("EXECUTING", 0),
        "Unique Blocked": unique_c.get("BLOCKED", 0),
        "Unique Descoped": unique_c.get("DESCOPED", 0),
        "Unique To Do": unique_c.get("TO DO", 0),
        "Pass % (of executed)": pass_pct,
        "RT Completion %": completion_pct,
        "RT Remaining %": remaining_pct,
        "Actual Execution (Raw Runs)": total_runs,
        "Duplicate Runs": dup_runs
    }

    file_exists = os.path.exists(fname)
    with open(fname, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            w.writeheader()
        w.writerow(row)
    return fname

csv_path = save_daily_snapshot(
    PLAN_KEY, planned_count, unique_counts, total_runs, duplicate_runs,
    pass_pct, completion_pct, remaining_pct
)
print(f"[+] Snapshot appended to {csv_path}\n")
